# 🚀 Mô hình Dự báo Sản xuất - Detrended XGBoost (Optimal)
**Dự án:** Datathon 2026 - Team Outliers  

**Phương pháp:** Detrended XGBoost - Đã cập nhật bộ tham số **Tối ưu nhất** tìm được bởi Optuna (MAE ~615k).

## 1. Khai báo thư viện và Cấu hình

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = '../Data/sales.csv'
SUBMISSION_SAMPLE_PATH = '../Data/sample_submission.csv'
OUTPUT_PATH = '../submission_v4_production_optimal.csv'
RANDOM_SEED = 42

## 2. Tải và Tiền xử lý dữ liệu

In [ ]:
df_train = pd.read_csv(TRAIN_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df_sample = pd.read_csv(SUBMISSION_SAMPLE_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df_train.tail()

## 3. Kỹ thuật tạo đặc trưng (Feature Engineering)

In [ ]:
TET_DATES = pd.to_datetime([
    '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08',
    '2017-01-28', '2018-02-16', '2019-02-05', '2020-01-25', '2021-02-12',
    '2022-02-01', '2023-01-22', '2024-02-10'
])

def create_features(df):
    out = df.copy()
    out['month'] = out['Date'].dt.month
    out['day'] = out['Date'].dt.day
    out['dayofweek'] = out['Date'].dt.dayofweek
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    out['is_holiday'] = 0
    for month, day in [(1, 1), (4, 30), (5, 1), (9, 2), (12, 25)]:
        out.loc[(out['month'] == month) & (out['day'] == day), 'is_holiday'] = 1
    out['days_to_tet'] = 999
    for tet in TET_DATES:
        diff = (out['Date'] - tet).dt.days
        mask = (diff >= -20) & (diff <= 10)
        out.loc[mask, 'days_to_tet'] = diff[mask]
    return out

## 4. Xây dựng mô hình Detrended XGBoost

In [ ]:
# Cập nhật bộ tham số TỐI ƯU NHẤT từ Optuna
XGB_PARAMS = {
    'n_estimators': 1334,
    'learning_rate': 0.0201745325919737,
    'max_depth': 5,
    'subsample': 0.6555722188096532,
    'colsample_bytree': 0.7322563076249518,
    'min_child_weight': 10,
    'random_state': RANDOM_SEED,
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
}

def fit_predict_detrended(train_df, future_dates, target_col, params):
    tr = train_df.copy()
    tr['year'] = tr['Date'].dt.year
    annual_mean = tr.groupby('year')[target_col].mean()
    tr['norm_target'] = tr[target_col] / tr['year'].map(annual_mean)
    
    x_train = create_features(tr)
    x_future = create_features(pd.DataFrame({'Date': pd.to_datetime(future_dates)}))
    feature_cols = ['month', 'day', 'dayofweek', 'is_weekend', 'is_holiday', 'days_to_tet']

    model = XGBRegressor(**params)
    model.fit(x_train[feature_cols], tr['norm_target'])

    if len(annual_mean) >= 2:
        growth = annual_mean.pct_change().dropna().mean() + 1
    else:
        growth = 1.0

    base_year = int(annual_mean.index.max())
    base_mean = float(annual_mean.loc[base_year])
    yearly_scale = base_mean * (growth ** (x_future['Date'].dt.year - base_year))
    
    pred = model.predict(x_future[feature_cols]) * yearly_scale
    return np.clip(pred, 0, None)

## 5. Đánh giá hiệu năng (Backtesting)
Kiểm tra với bộ tham số mới trên năm 2022.

In [ ]:
HORIZON = 548
split_date = df_train['Date'].max() - pd.Timedelta(days=HORIZON)
train_split = df_train[df_train['Date'] <= split_date]
valid_split = df_train[df_train['Date'] > split_date]

rev_val_pred = fit_predict(train_split, valid_split['Date'], 'Revenue')
cogs_val_pred = fit_predict(train_split, valid_split['Date'], 'COGS')

print(f"Revenue: MAE = {mean_absolute_error(valid_split['Revenue'], rev_val_pred):,.0f}")
print(f"COGS:    MAE = {mean_absolute_error(valid_split['COGS'], cogs_val_pred):,.0f}")

plt.figure(figsize=(15, 6))
plt.plot(valid_split['Date'], valid_split['Revenue'], label='Thực tế', color='black', alpha=0.3)
plt.plot(valid_split['Date'], rev_val_pred, label='Dự báo (Optimal XGB)', color='red', linestyle='--')
plt.title('Revenue: Thực tế vs Dự báo (Backtest)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 6. Dự báo tương lai và Xuất file Submission

In [ ]:
rev_final = fit_predict_detrended(df_train, df_sample['Date'], 'Revenue', XGB_PARAMS)
cogs_final = fit_predict_detrended(df_train, df_sample['Date'], 'COGS', XGB_PARAMS)

submission = df_sample.copy()
submission['Revenue'] = rev_final
submission['COGS'] = cogs_final

plt.figure(figsize=(15, 5))
plt.plot(df_train['Date'].tail(120), df_train['Revenue'].tail(120), label='Thực tế (Cuối 2022)', color='steelblue')
plt.plot(submission['Date'].head(120), submission['Revenue'].head(120), label='Dự báo (Đầu 2023)', color='tomato', linestyle='--')
plt.title('Kiểm tra chuyển tiếp (Revenue)')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission.to_csv(OUTPUT_PATH, index=False)
print(f"[+] File submission TỐI ƯU đã lưu tại: {OUTPUT_PATH}")

## 7. Trực quan hóa kết quả (Interactive Visualization)

In [ ]:
def plot_forecast_plotly(df_train, df_forecast, target_name):
    fig = go.Figure()
    df_last_year = df_train.tail(365)
    fig.add_trace(go.Scatter(x=df_last_year['Date'], y=df_last_year[target_name], name='Thực tế 2022'))
    fig.add_trace(go.Scatter(x=df_forecast['Date'], y=df_forecast[target_name], name='Dự báo 2023-2024', line=dict(dash='dot')))
    fig.update_layout(title=f'Dự báo {target_name} (Optimal)', template="plotly_white", hovermode="x unified")
    fig.show()

plot_forecast_plotly(df_train, submission, 'Revenue')
plot_forecast_plotly(df_train, submission, 'COGS')